In [1]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset,DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

device='cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
data_df=pd.read_csv('processed_vulcan_gesture_data.csv')
data_df.dropna(inplace=True)
data_df.drop(['label'],axis=1,inplace=True)
data_df.drop(data_df.columns[0:3],axis=1,inplace=True)
data_df.head()

,x2,y2,z2,x3,y3,z3,x4,y4,z4,x5,y5,z5,x6,y6,z6,x7,y7,z7,x8,y8,z8,x9,y9,z9,x10,y10,z10,x11,y11,z11,x12,y12,z12,x13,y13,z13,x14,y14,z14,x15,y15,z15,x16,y16,z16,x17,y17,z17,x18,y18,z18,x19,y19,z19,x20,y20,z20,x21,y21,z21
0,0.301471,-0.116610,-0.129769,0.553936,-0.335465,-0.215508,0.755322,-0.507845,-0.295683,0.945360,-0.640039,-0.381784,0.396121,-0.918198,-0.185651,0.512111,-1.323311,-0.304754,0.576883,-1.578097,-0.385526,0.626304,-1.819376,-0.446320,0.216408,-0.999652,-0.206703,0.348625,-1.443307,-0.329915,0.449328,-1.712919,-0.424566,0.528836,-1.954375,-0.491130,0.018328,-0.959002,-0.239267,-0.012191,-1.412573,-0.366489,-0.023144,-1.721060,-0.460416,-0.016109,-1.995845,-0.521810,-0.175887,-0.828014,-0.279678,-0.199188,-1.195211,-0.394022,-0.199216,-1.427827,-0.456839,-0.182843,-1.658963,-0.498779
1,0.310695,-0.097785,-0.143740,0.571911,-0.303764,-0.241140,0.784800,-0.481705,-0.331246,0.983201,-0.636188,-0.425857,0.421239,-0.906950,-0.206668,0.551793,-1.316823,-0.335870,0.625584,-1.577068,-0.419152,0.681641,-1.823068,-0.481540,0.244179,-1.006786,-0.227552,0.385554,-1.455754,-0.370362,0.488531,-1.749340,-0.474145,0.565244,-2.012833,-0.544906,0.041122,-0.972685,-0.260457,0.012063,-1.449440,-0.409063,-0.010275,-1.767215,-0.508699,-0.011348,-2.053308,-0.571235,-0.161902,-0.843127,-0.300844,-0.180331,-1.205614,-0.432912,-0.174111,-1.438307,-0.503087,-0.148426,-1.671955,-0.548812
2,0.298582,-0.127836,-0.136238,0.541911,-0.346281,-0.222393,0.741670,-0.541025,-0.299983,0.923229,-0.682583,-0.380292,0.338825,-0.940849,-0.179774,0.409319,-1.358869,-0.283332,0.448496,-1.609162,-0.352256,0.477369,-1.846969,-0.401708,0.144469,-1.006466,-0.186630,0.224057,-1.455988,-0.287558,0.296861,-1.726613,-0.357173,0.351426,-1.962097,-0.403327,-0.052833,-0.958992,-0.206293,-0.130434,-1.387123,-0.312110,-0.170561,-1.673601,-0.383841,-0.189105,-1.924000,-0.427618,-0.234256,-0.828470,-0.236035,-0.283996,-1.175345,-0.326966,-0.309019,-1.389551,-0.371651,-0.314884,-1.599180,-0.398628
3,0.291528,-0.112419,-0.165270,0.535371,-0.336183,-0.263411,0.734568,-0.530090,-0.348441,0.920105,-0.669224,-0.433564,0.358018,-0.933715,-0.205885,0.418154,-1.361383,-0.322225,0.435311,-1.630831,-0.401619,0.437064,-1.883040,-0.456435,0.168660,-1.003385,-0.201201,0.244221,-1.447543,-0.309302,0.283766,-1.731766,-0.382122,0.298146,-1.991632,-0.428935,-0.030291,-0.967435,-0.211799,-0.096175,-1.381480,-0.322898,-0.158656,-1.645819,-0.387500,-0.209094,-1.881762,-0.422543,-0.224722,-0.857689,-0.235741,-0.278867,-1.198789,-0.325507,-0.313692,-1.398753,-0.360377,-0.334747,-1.591777,-0.377276
4,0.276684,-0.138026,-0.128775,0.493676,-0.381055,-0.198053,0.669122,-0.596571,-0.258636,0.831140,-0.746427,-0.318719,0.295459,-0.955355,-0.133473,0.348660,-1.355276,-0.221698,0.364579,-1.603786,-0.287263,0.368613,-1.838751,-0.333445,0.119229,-1.010935,-0.130681,0.188637,-1.440889,-0.209586,0.227448,-1.713921,-0.268898,0.246395,-1.955682,-0.307321,-0.058601,-0.961729,-0.142086,-0.129564,-1.342335,-0.222225,-0.180235,-1.598498,-0.276044,-0.216446,-1.825579,-0.307583,-0.225422,-0.840431,-0.165610,-0.277751,-1.152860,-0.230830,-0.306111,-1.343216,-0.257092,-0.318585,-1.530783,-0.270657


In [9]:
d_copy=data_df.copy()
for column in data_df.columns:
    data_df[column]=data_df[column]/data_df[column].abs().max()
data_df.head()

,x2,y2,z2,x3,y3,z3,x4,y4,z4,x5,y5,z5,x6,y6,z6,x7,y7,z7,x8,y8,z8,x9,y9,z9,x10,y10,z10,x11,y11,z11,x12,y12,z12,x13,y13,z13,x14,y14,z14,x15,y15,z15,x16,y16,z16,x17,y17,z17,x18,y18,z18,x19,y19,z19,x20,y20,z20,x21,y21,z21
0,0.970314,-0.844837,-0.785191,0.968570,-0.880359,-0.818143,0.962439,-0.851274,-0.848591,0.961513,-0.857471,-0.880569,0.940371,-0.961107,-0.898306,0.928084,-0.972034,-0.907358,0.922151,-0.967664,-0.919776,0.918819,-0.966191,-0.926860,0.886266,-0.988840,-0.908376,0.904218,-0.99129,-0.890790,0.919755,-0.979180,-0.895434,0.935588,-0.970957,-0.901311,0.312768,-0.985933,-0.918644,-0.093468,-0.974565,-0.895921,-0.128411,-0.973883,-0.905085,-0.074423,-0.972014,-0.913477,-0.750833,-0.965401,-0.929643,-0.701375,-0.991371,-0.910168,-0.635070,-0.992714,-0.908070,-0.546214,-0.992230,-0.908834
1,1.000000,-0.708455,-0.869725,1.000000,-0.797164,-0.915452,1.000000,-0.807456,-0.950653,1.000000,-0.852312,-0.982224,1.000000,-0.949332,-1.000000,1.000000,-0.967269,-1.000000,1.000000,-0.967033,-1.000000,1.000000,-0.968152,-1.000000,1.000000,-0.995896,-1.000000,1.000000,-0.99984,-1.000000,1.000000,-1.000000,-1.000000,1.000000,-1.000000,-1.000000,0.701727,-1.000000,-1.000000,0.092481,-1.000000,-1.000000,-0.057011,-1.000000,-1.000000,-0.052430,-1.000000,-1.000000,-0.691132,-0.983022,-1.000000,-0.634978,-1.000000,-1.000000,-0.555038,-1.000000,-1.000000,-0.443398,-1.000000,-1.000000
2,0.961014,-0.926172,-0.824334,0.947545,-0.908743,-0.844282,0.945043,-0.906892,-0.860931,0.939003,-0.914468,-0.877129,0.804353,-0.984816,-0.869870,0.741797,-0.998154,-0.843578,0.716924,-0.986713,-0.840404,0.700323,-0.980844,-0.834215,0.591650,-0.995579,-0.820163,0.581132,-1.00000,-0.776422,0.607660,-0.987008,-0.753299,0.621725,-0.974794,-0.740176,-0.901582,-0.985922,-0.792044,-1.000000,-0.957006,-0.762987,-0.946326,-0.947027,-0.754554,-0.873681,-0.937025,-0.748585,-1.000000,-0.965933,-0.784575,-1.000000,-0.974893,-0.755272,-0.985103,-0.966102,-0.738741,-0.940663,-0.956473,-0.726347
3,0.938309,-0.814474,-1.000000,0.936110,-0.882241,-1.000000,0.935994,-0.888562,-1.000000,0.935826,-0.896570,-1.000000,0.849917,-0.977348,-0.996211,0.757809,-1.000000,-0.959373,0.695847,-1.000000,-0.958172,0.641195,-1.000000,-0.947864,0.690723,-0.992532,-0.884198,0.633429,-0.99420,-0.835133,0.580855,-0.989954,-0.805917,0.527464,-0.989467,-0.787171,-0.516912,-0.994603,-0.813184,-0.737341,-0.953113,-0.789360,-0.880272,-0.931307,-0.761747,-0.966033,-0.916454,-0.739700,-0.959301,-1.000000,-0.783598,-0.981939,-0.994339,-0.751902,-1.000000,-0.972500,-0.716331,-1.000000,-0.952045,-0.687441
4,0.890533,-1.000000,-0.779178,0.863204,-1.000000,-0.751876,0.852601,-1.000000,-0.742267,0.845341,-1.000000,-0.735114,0.701405,-1.000000,-0.645834,0.631867,-0.995514,-0.660072,0.582782,-0.983416,-0.685343,0.540773,-0.976480,-0.692454,0.488286,-1.000000,-0.574290,0.489262,-0.98963,-0.565895,0.465575,-0.979753,-0.567123,0.435908,-0.971607,-0.563988,-1.000000,-0.988736,-0.545527,-0.993325,-0.926106,-0.543254,-1.000000,-0.904530,-0.542646,-1.000000,-0.889092,-0.538453,-0.962291,-0.979878,-0.550484,-0.978010,-0.956243,-0.533202,-0.975831,-0.933887,-0.511027,-0.951720,-0.915565,-0.493168
